In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report
import time

In [2]:
BATCH_SIZE = 2048 # Increased for speed
LEARNING_RATE = 0.001
EPOCHS = 50
RFF_COMPONENTS = 2048 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

Using device: cuda


In [3]:
def load_and_preprocess():
    print("Loading UNSW-NB15 files...")
    # These CSVs usually have headers, so we don't define names manually
    train_df = pd.read_csv('UNSW_NB15_training-set.csv')
    test_df = pd.read_csv('UNSW_NB15_testing-set.csv')

    # --- REQUESTED: PRINT FIRST 5 ROWS ---
    print("\n--- First 5 Rows of Raw Training Data ---")
    print(train_df.head())
    print("-" * 50)

    # 1. Clean useless columns
    # 'id' is just a row number
    # 'attack_cat' is the specific attack name (we only want binary 'label')
    drop_cols = ['id', 'attack_cat']
    train_df.drop([c for c in drop_cols if c in train_df.columns], axis=1, inplace=True)
    test_df.drop([c for c in drop_cols if c in test_df.columns], axis=1, inplace=True)

    # 2. Separate Target (y) and Features (X)
    y_train = train_df['label'].values
    y_test = test_df['label'].values
    
    X_train = train_df.drop('label', axis=1)
    X_test = test_df.drop('label', axis=1)

    # 3. Identify columns automatically
    # UNSW has 'proto', 'service', 'state' as strings (object)
    cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
    num_cols = X_train.select_dtypes(exclude=['object']).columns.tolist()
    
    print(f"Categorical Columns found: {cat_cols}")
    print(f"Numerical Columns found: {len(num_cols)}")

    # 4. Pipeline
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
        ]
    )

    print("Fitting preprocessor (this handles the different protocols/services)...")
    X_train_enc = preprocessor.fit_transform(X_train).astype(np.float32)
    X_test_enc = preprocessor.transform(X_test).astype(np.float32)

    return X_train_enc, y_train, X_test_enc, y_test

In [4]:
class RBFSVM(nn.Module):
    def __init__(self, input_dim, rff_dim=2048, sigma=1.0):
        super(RBFSVM, self).__init__()
        self.rff_weights = nn.Parameter(torch.randn(input_dim, rff_dim) / sigma, requires_grad=False)
        self.rff_bias = nn.Parameter(torch.rand(rff_dim) * 2 * np.pi, requires_grad=False)
        self.fc = nn.Linear(rff_dim, 1)
        
    def forward(self, x):
        x_mapped = torch.cos(x @ self.rff_weights + self.rff_bias)
        return self.fc(x_mapped)

In [5]:
X_train, y_train, X_test, y_test = load_and_preprocess()

# Convert targets to -1 / 1 for Hinge Loss
y_train_svm = torch.tensor(np.where(y_train == 0, -1, 1), dtype=torch.float32).to(device)

X_train_tensor = torch.tensor(X_train).to(device)
X_test_tensor = torch.tensor(X_test).to(device)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_svm), 
                            batch_size=BATCH_SIZE, shuffle=True)

input_dim = X_train.shape[1]
model = RBFSVM(input_dim, rff_dim=RFF_COMPONENTS, sigma=2.0).to(device)
optimizer = optim.Adam(model.fc.parameters(), lr=LEARNING_RATE, weight_decay=0.001)

print(f"\nStarting Training on UNSW-NB15...")
start_time = time.time()

model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X).squeeze()
        loss = torch.mean(torch.clamp(1 - batch_y * outputs, min=0))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {total_loss/len(train_loader):.4f}")

print(f"Training Time: {time.time() - start_time:.2f}s")

Loading UNSW-NB15 files...

--- First 5 Rows of Raw Training Data ---
   id       dur proto service state  spkts  dpkts  sbytes  dbytes       rate  \
0   1  0.121478   tcp       -   FIN      6      4     258     172  74.087490   
1   2  0.649902   tcp       -   FIN     14     38     734   42014  78.473372   
2   3  1.623129   tcp       -   FIN      8     16     364   13186  14.170161   
3   4  1.681642   tcp     ftp   FIN     12     12     628     770  13.677108   
4   5  0.449454   tcp       -   FIN     10      6     534     268  33.373826   

   ...  ct_dst_sport_ltm  ct_dst_src_ltm  is_ftp_login  ct_ftp_cmd  \
0  ...                 1               1             0           0   
1  ...                 1               2             0           0   
2  ...                 1               3             0           0   
3  ...                 1               3             1           1   
4  ...                 1              40             0           0   

   ct_flw_http_mthd  ct_src_

In [6]:
model.eval()
with torch.no_grad():
    outputs = model(X_test_tensor).squeeze()
    predictions = torch.where(outputs >= 0, 1, 0).cpu().numpy()

print("\n--- UNSW-NB15 Evaluation Results ---")
print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")
print(classification_report(y_test, predictions, target_names=['Normal', 'Attack']))


--- UNSW-NB15 Evaluation Results ---
Accuracy: 0.8416
              precision    recall  f1-score   support

      Normal       0.97      0.67      0.79     37000
      Attack       0.78      0.99      0.87     45332

    accuracy                           0.84     82332
   macro avg       0.88      0.83      0.83     82332
weighted avg       0.87      0.84      0.84     82332

